# Chapter 10 Tutorial — Markov Renewal Theory

This notebook is a step-by-step tutorial for the main concepts in Chapter 10: **Markov renewal processes**, **semi-Markov processes**, **Markov renewal functions**, **classification of states**, **Markov renewal equations**, and **limit theorems**.

The goal is not just to list formulas. The goal is to build the working mental model:

> A Markov renewal process is a Markov chain whose transitions carry random holding times.  
> The next state depends only on the current state, and the distribution of the next holding time may depend on both the current and next state.

We will use code to simulate examples, solve matrix renewal equations numerically, and visualize semi-Markov sample paths.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import inv
from scipy.integrate import quad
from scipy.stats import expon, gamma

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)

## 1. From Markov chains and renewal processes to Markov renewal processes

Two earlier objects are being combined.

### Renewal process

A renewal process has interarrival times

\[
W_1, W_2, \dots
\]

which are independent and identically distributed. The renewal times are

\[
S_0 = 0,\qquad S_n = W_1+\cdots+W_n.
\]

The future after a renewal time restarts probabilistically.

### Markov chain

A Markov chain has states

\[
X_0, X_1, X_2,\dots
\]

and transition probabilities

\[
P(i,j)=P(X_{n+1}=j\mid X_n=i).
\]

The future state depends only on the present state.

### Markov renewal process

A Markov renewal process tracks both:

\[
(X_n,T_n), \qquad n\in \mathbb N,
\]

where \(X_n\) is the state visited at the \(n\)-th transition, and \(T_n\) is the time of that transition.

The increments

\[
T_{n+1}-T_n
\]

are no longer identically distributed in general. Their law may depend on the transition \(i\to j\).

## 2. Definition

A stochastic process

\[
(X,T)=\{(X_n,T_n): n\in\mathbb N\}
\]

is a **Markov renewal process** with state space \(E\) if

\[
P\{X_{n+1}=j,\; T_{n+1}-T_n \le t
\mid X_0,\dots,X_n;\;T_0,\dots,T_n\}
=
P\{X_{n+1}=j,\; T_{n+1}-T_n \le t\mid X_n\}.
\]

In words:

> Given the current state \(X_n\), the joint distribution of the next state and the next holding time does not depend on the earlier path.

For a time-homogeneous Markov renewal process, define the **semi-Markov kernel**

\[
Q(i,j,t)
=
P_i\{X_1=j,\; T_1\le t\}.
\]

This is a matrix of subdistribution functions in \(t\).

## 3. The embedded Markov chain

The next-state transition matrix is obtained by letting \(t\to\infty\):

\[
P(i,j)=Q(i,j,\infty).
\]

So \(\{X_n\}\) is an ordinary Markov chain with transition matrix \(P\).

For fixed \(i\),

\[
\sum_j P(i,j)=1.
\]

The semi-Markov kernel \(Q\) contains more information than \(P\): it tells us **both** where the process jumps and how long the jump takes.

In [ ]:
# Example embedded transition matrix
P = np.array([
    [0.2, 0.8],
    [0.6, 0.4],
])
P.sum(axis=1)

## 4. Conditional holding-time distributions

If \(P(i,j)>0\), define

\[
G(i,j,t)=\frac{Q(i,j,t)}{P(i,j)}.
\]

This is the conditional distribution of the holding time, given that the next transition is \(i\to j\):

\[
G(i,j,t)
=
P_i\{T_1\le t \mid X_1=j\}.
\]

Thus the semi-Markov kernel factors as

\[
Q(i,j,t)=P(i,j)G(i,j,t).
\]

### Thinking model

A transition has two parts:

1. choose the next state \(j\) using \(P(i,j)\);
2. sample a holding time from \(G(i,j,\cdot)\).

## 5. Important conditional independence result

For any \(n\), the increments

\[
T_1-T_0,\quad T_2-T_1,\quad \dots,\quad T_n-T_{n-1}
\]

are conditionally independent given the visited states

\[
X_0,X_1,\dots,X_n.
\]

Moreover,

\[
P_i\{T_1-T_0\le u_1,\dots,T_n-T_{n-1}\le u_n
\mid X_0,\dots,X_n\}
=
\prod_{m=0}^{n-1}G(X_m,X_{m+1},u_{m+1}).
\]

This is the continuous-time analogue of the Markov property: the path of states controls the distributions of the holding times, but after conditioning on the state path, the holding times split apart.

## 6. Simulating a Markov renewal process

We will use a two-state system. The embedded chain has transition matrix

\[
P=
\begin{pmatrix}
0.2 & 0.8\\
0.6 & 0.4
\end{pmatrix}.
\]

Let the holding times be exponential, but with rates depending on the transition:

\[
\lambda_{00}=2,\quad
\lambda_{01}=0.7,\quad
\lambda_{10}=1.5,\quad
\lambda_{11}=0.4.
\]

A small rate means a longer average holding time.

In [ ]:
P = np.array([
    [0.2, 0.8],
    [0.6, 0.4],
])

rates = np.array([
    [2.0, 0.7],
    [1.5, 0.4],
])

def simulate_markov_renewal(P, rates, n_steps=30, x0=0, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    states = [x0]
    times = [0.0]
    holding_times = []
    for _ in range(n_steps):
        i = states[-1]
        j = rng.choice(len(P), p=P[i])
        w = rng.exponential(1 / rates[i, j])
        states.append(j)
        holding_times.append(w)
        times.append(times[-1] + w)
    return np.array(states), np.array(times), np.array(holding_times)

states, times, holding = simulate_markov_renewal(P, rates, n_steps=40, x0=0, rng=rng)

states[:10], times[:10], holding[:10]

In [ ]:
plt.figure(figsize=(10, 3))
plt.step(times, states, where="post")
plt.scatter(times, states)
plt.xlabel("time")
plt.ylabel("state")
plt.yticks([0, 1])
plt.title("Sample path of a two-state semi-Markov process")
plt.show()

## 7. The associated semi-Markov process

From a Markov renewal process \((X_n,T_n)\), define a continuous-time process

\[
Y_t = X_n
\quad\text{if}\quad
T_n \le t < T_{n+1}.
\]

This is called the **minimal semi-Markov process** associated with \((X,T)\).

In words:

> \(Y_t\) is the state occupied at clock time \(t\), while \(X_n\) is the state at the \(n\)-th jump.

The Markov renewal process describes the jump chain and jump times. The semi-Markov process fills in the state between jumps.

In [ ]:
# Estimate fraction of time spent in each state from one long run.
states_long, times_long, holding_long = simulate_markov_renewal(P, rates, n_steps=200_000, x0=0, rng=rng)

time_in_state = np.zeros(2)
for n, w in enumerate(holding_long):
    time_in_state[states_long[n]] += w

time_fraction = time_in_state / time_in_state.sum()
time_fraction

The time fractions are not the same as the stationary distribution of the embedded Markov chain, because states can have different mean sojourn times.

The embedded chain tells us **how often** states are visited.

The semi-Markov process asks **how much clock time** is spent in each state.

In [ ]:
# Stationary distribution of embedded chain: solve pi = pi P, sum pi = 1
A = np.vstack([P.T - np.eye(2), np.ones(2)])
b = np.array([0, 0, 1], dtype=float)
pi_embedded, *_ = np.linalg.lstsq(A, b, rcond=None)

# Mean sojourn time in state i:
# m_i = sum_j P(i,j) E[holding | i -> j] = sum_j P(i,j) / rate(i,j)
m = (P / rates).sum(axis=1)
pi_time = pi_embedded * m
pi_time = pi_time / pi_time.sum()

pi_embedded, m, pi_time, time_fraction

The theoretical long-run time distribution is

\[
\pi_i^{\text{time}}
=
\frac{\pi_i m_i}{\sum_k \pi_k m_k},
\]

where \(\pi\) is the invariant distribution of the embedded chain and

\[
m_i = E_i[T_1]
     = \sum_j P(i,j)\,E_i[T_1\mid X_1=j].
\]

## 8. Examples from the chapter

### Example: Markov processes

If the holding time from state \(i\) is exponential with parameter \(\lambda(i)\), independent of the next state, then

\[
Q(i,j,t)=P(i,j)(1-e^{-\lambda(i)t}).
\]

The associated semi-Markov process is a continuous-time Markov process.

The exponential law is special because of memorylessness. Without exponential holding times, the process usually remembers the age of the current sojourn and is not Markov in clock time.

### Example: traffic theory

The state can describe the type of a vehicle, say car or truck, and the holding time is the time until the next arrival. The type of the next vehicle and the gap until its arrival may be statistically dependent.

### Example: counter of type 1 particles

Particles arrive according to a Poisson process. Each particle changes the counter state depending on whether it is type 1 or not. The embedded states are counter states, and jump times are arrival times.

### Example: \(M/G/1\) queue

The queue length observed after departures forms a Markov renewal structure. The time between embedded epochs is a service time plus arrivals during service. This is a standard way to analyze queue sizes at departure epochs.

## 9. Markov renewal functions

Define

\[
Q^{(n)}(i,j,t)
=
P_i\{X_n=j,\;T_n\le t\}.
\]

The **Markov renewal function** is

\[
R(i,j,t)
=
\sum_{n=0}^{\infty} Q^{(n)}(i,j,t).
\]

Interpretation:

\[
R(i,j,t)
=
E_i\left[\sum_{n=0}^{\infty}1_{\{X_n=j,\;T_n\le t\}}\right].
\]

So \(R(i,j,t)\) is the expected number of visits to state \(j\) by renewal time \(t\), starting from state \(i\).

The initial term is

\[
Q^{(0)}(i,j,t)=I(i,j),
\]

for \(t\ge 0\).

## 10. Recursive relation for \(Q^{(n)}\)

For \(n\ge 0\),

\[
Q^{(n+1)}(i,k,t)
=
\sum_j \int_0^t Q(i,j,ds)\,Q^{(n)}(j,k,t-s).
\]

This is the Markov renewal analogue of convolution.

Thinking model:

> To reach \(k\) by the \((n+1)\)-st jump by time \(t\), first jump from \(i\) to some \(j\) at time \(s\), then reach \(k\) from \(j\) in \(n\) more jumps by remaining time \(t-s\).

## 11. Laplace transforms

The chapter uses Laplace transforms to turn renewal convolutions into matrix algebra. Define

\[
Q_\alpha(i,j)=\int_0^\infty e^{-\alpha t}Q(i,j,dt).
\]

Then

\[
Q^{(n)}_\alpha = Q_\alpha^n.
\]

Hence

\[
R_\alpha
=
I+Q_\alpha+Q_\alpha^2+\cdots.
\]

If the state space is finite and the inverse exists,

\[
R_\alpha=(I-Q_\alpha)^{-1}.
\]

This is one of the most useful computational formulas in the chapter.

In [ ]:
# For exponential holding times:
# Q_alpha(i,j) = P(i,j) * E[e^{-alpha W_ij}]
# If W_ij ~ Exp(rate_ij), transform is rate/(rate+alpha)

def Q_alpha(P, rates, alpha):
    return P * (rates / (rates + alpha))

for alpha in [0.1, 0.5, 1.0, 2.0]:
    Qa = Q_alpha(P, rates, alpha)
    Ra = np.linalg.inv(np.eye(2) - Qa)
    print("alpha =", alpha)
    print("Q_alpha =")
    print(Qa)
    print("R_alpha =")
    print(Ra)
    print()

## 12. Classification of states

A state \(j\) is classified as recurrent or transient, and periodic or aperiodic, in the Markov renewal process.

A key point from the chapter:

> Recurrence and transience reduce to the same question for the embedded Markov chain \(X_n\).

If \(j\) is recurrent for the embedded chain, it is recurrent for the Markov renewal process. If it is transient for the embedded chain, it is transient for the Markov renewal process.

Periodicity is subtler: it depends not only on the state transitions, but also on the possible times at which returns can occur.

The chapter proves that if states \(i\) and \(j\) communicate, then they are either both aperiodic or both periodic, and in the periodic case they have the same period.

## 13. Markov renewal equations

Let \(\mathcal B\) be the class of bounded functions \(f(i,t)\), and define

\[
(Q*f)(i,t)
=
\sum_j \int_0^t Q(i,j,ds)\,f(j,t-s).
\]

A **Markov renewal equation** has the form

\[
f = g + Q*f.
\]

The solution is

\[
f = R*g,
\]

where

\[
(R*g)(i,t)
=
\sum_j \int_0^t R(i,j,ds)\,g(j,t-s).
\]

### Proof idea

Repeated substitution gives

\[
f
=
g + Q*g + Q^{(2)}*g+\cdots + Q^{(n)}*g + Q^{(n+1)}*f.
\]

If the remainder term goes away, then

\[
f=\sum_{n=0}^{\infty}Q^{(n)}*g=R*g.
\]

## 14. Non-uniqueness and harmonic solutions

The equation

\[
f = g + Q*f
\]

does not always have a unique bounded solution. The chapter states that every solution has the form

\[
f = R*g + h,
\]

where

\[
h=Q*h.
\]

Thus \(h\) is a harmonic term.

For finite state spaces, the harmonic term vanishes under the right transience/discounting conditions, and the solution is unique.

## 15. Criteria for uniqueness

The chapter studies the equation

\[
h = Q*h,\qquad 0\le h\le 1.
\]

There is a nonzero bounded solution if and only if the Markov renewal process can avoid renewal forever in a certain sense.

A key condition uses

\[
L=\sup_n T_n.
\]

If

\[
L=+\infty
\]

almost surely, then the only solution of

\[
h=Q*h,\quad 0\le h\le 1
\]

is \(h=0\).

Thinking model:

> If time keeps going to infinity through renewal epochs, then no nonzero bounded mass can hide forever in the tail term.

## 16. Sufficient conditions for \(L=\infty\)

Several results in the chapter give conditions implying

\[
L=+\infty \quad \text{almost surely}.
\]

Examples:

1. If all states are recurrent, then \(L=\infty\) almost surely.
2. If there are only finitely many transient states, then \(L=\infty\) almost surely.
3. A uniform positive lower-bound condition on sojourn-time tails also implies \(L=\infty\).
4. If there is a transient set \(A\) with finite expected total time spent in \(A\), and a positive probability to leave \(A\), then \(L\) can be finite with positive probability.

This part is the Markov renewal analogue of determining whether the process can keep renewing forever.

## 17. Limit theorems: recurrent aperiodic case

For fixed \(i,j\), \(R(i,j,\cdot)\) is a possibly delayed renewal function. Therefore, its limiting behavior is inherited from renewal theory.

If \(j\) is recurrent and aperiodic, then

\[
\lim_{t\to\infty}
\left[
R(i,j,t+a)-R(i,j,t)
\right]
=
F_{ij}(\infty)\,m(j)^{-1}\,a,
\]

where:

- \(F_{ij}(\infty)\) is the probability of ever reaching \(j\) from \(i\),
- \(m(j)\) is the mean recurrence time of state \(j\) in the Markov renewal process.

For recurrent irreducible processes, \(F_{ij}(\infty)=1\), so the asymptotic renewal rate at state \(j\) is

\[
\frac{1}{m(j)}.
\]

## 18. Mean recurrence time

Let \(v\) be an invariant measure for the embedded Markov chain:

\[
v = vP.
\]

The chapter gives a formula for the inverse mean recurrence time:

\[
\frac{1}{m(i)}
=
\frac{v(i)}{v m},
\]

where

\[
m(i)
=
E_i[T_1]
=
\sum_j \int_0^\infty t\,Q(i,j,dt),
\]

and

\[
vm=\sum_j v(j)m(j).
\]

If \(v\) is normalized as the stationary distribution \(\pi\), then

\[
\frac{1}{m(i)}
=
\frac{\pi_i}{\sum_j \pi_j m(j)}.
\]

This matches the long-run time-fraction calculation earlier.

In [ ]:
# Verify mean recurrence-time formula numerically for the two-state example.

# Embedded stationary distribution
A = np.vstack([P.T - np.eye(2), np.ones(2)])
b = np.array([0, 0, 1], dtype=float)
pi, *_ = np.linalg.lstsq(A, b, rcond=None)

# Mean sojourn by state
m_state = (P / rates).sum(axis=1)
mean_cycle_denominator = np.dot(pi, m_state)
inverse_mean_recurrence = pi / mean_cycle_denominator
mean_recurrence = 1 / inverse_mean_recurrence

print("embedded stationary pi:", pi)
print("mean sojourn m(i):", m_state)
print("long-run time arrival rate into state i:", inverse_mean_recurrence)
print("mean recurrence time to state i:", mean_recurrence)

## 19. Limit theorem for \(R*g\)

A central limit result in the chapter says that for an irreducible aperiodic Markov renewal process with finite state space,

\[
\lim_{t\to\infty}
\sum_j\int_0^t R(i,j,ds)\,g(j,t-s)
=
\frac{1}{vm}
\sum_j v(j)\int_0^\infty g(j,s)\,ds,
\]

under direct Riemann integrability assumptions.

Interpretation:

> In the long run, the initial state \(i\) is forgotten. The answer is a weighted average over states using the invariant measure of the embedded chain and the mean clock-time scale.

## 20. Worked numerical example: solving a discounted Markov renewal equation

Suppose

\[
f = g + Q*f.
\]

A discounted version uses Laplace transforms. For exponential transition times, the transform equation is

\[
\hat f_\alpha = \hat g_\alpha + Q_\alpha \hat f_\alpha,
\]

so

\[
\hat f_\alpha = (I-Q_\alpha)^{-1}\hat g_\alpha.
\]

Let \(g(i,t)=c_i e^{-\beta t}\). Then

\[
\hat g_\alpha(i)=\int_0^\infty e^{-\alpha t}c_i e^{-\beta t}\,dt
=
\frac{c_i}{\alpha+\beta}.
\]

In [ ]:
c = np.array([10.0, 3.0])
beta = 0.6

for alpha in [0.1, 0.5, 1.0]:
    Qa = Q_alpha(P, rates, alpha)
    ghat = c / (alpha + beta)
    fhat = np.linalg.solve(np.eye(2) - Qa, ghat)
    print(f"alpha={alpha}: f_hat={fhat}")

## 21. Simulation check: occupation rate

A practical way to understand the limit theorem is to simulate a long semi-Markov path and estimate

\[
\frac{1}{t}\int_0^t 1_{\{Y_s=i\}}\,ds.
\]

This should converge to the long-run time fraction

\[
\frac{\pi_i m_i}{\sum_k \pi_k m_k}.
\]

In [ ]:
def occupation_time_estimates(P, rates, horizons, x0=0, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    max_horizon = max(horizons)
    states = [x0]
    times = [0.0]
    while times[-1] < max_horizon:
        i = states[-1]
        j = rng.choice(len(P), p=P[i])
        w = rng.exponential(1 / rates[i, j])
        states.append(j)
        times.append(times[-1] + w)

    states = np.array(states)
    times = np.array(times)
    estimates = []
    for H in horizons:
        occ = np.zeros(len(P))
        for n in range(len(times) - 1):
            start = times[n]
            end = min(times[n+1], H)
            if end > start:
                occ[states[n]] += end - start
            if times[n+1] >= H:
                break
        estimates.append(occ / H)
    return np.array(estimates)

horizons = np.array([10, 30, 100, 300, 1000, 3000, 10000])
estimates = occupation_time_estimates(P, rates, horizons, rng=rng)

plt.figure(figsize=(8, 4))
plt.plot(horizons, estimates[:, 0], marker="o", label="state 0 estimate")
plt.plot(horizons, estimates[:, 1], marker="o", label="state 1 estimate")
plt.axhline(pi_time[0], linestyle="--", label="state 0 theory")
plt.axhline(pi_time[1], linestyle="--", label="state 1 theory")
plt.xscale("log")
plt.xlabel("time horizon")
plt.ylabel("occupation fraction")
plt.title("Convergence of occupation fractions")
plt.legend()
plt.show()

pi_time

## 22. Connections to earlier chapters

Chapter 10 combines several earlier ideas:

| Earlier topic | Chapter 10 role |
|---|---|
| Renewal process | special case with one state |
| Markov chain | embedded state process \(X_n\) |
| Semi-Markov process | continuous-time process \(Y_t\) built from \((X_n,T_n)\) |
| Potential matrix | \(R=\sum Q^{(n)}\), now time-dependent |
| Renewal equation | becomes \(f=g+Q*f\) with a matrix kernel |
| Limit theorem | long-run behavior uses invariant measure and mean sojourn times |

The main conceptual upgrade is:

> Renewal theory restarts at renewal times.  
> Markov renewal theory restarts at renewal times with a remembered state.

## 23. Summary of key formulas

### Semi-Markov kernel

\[
Q(i,j,t)=P_i\{X_1=j,\;T_1\le t\}
\]

### Embedded transition matrix

\[
P(i,j)=Q(i,j,\infty)
\]

### Conditional holding-time law

\[
G(i,j,t)=\frac{Q(i,j,t)}{P(i,j)}
\]

### Markov renewal function

\[
R(i,j,t)=\sum_{n=0}^{\infty} Q^{(n)}(i,j,t)
\]

### Renewal equation

\[
f=g+Q*f
\]

### Canonical solution

\[
f=R*g
\]

### Laplace-transform potential

\[
R_\alpha=(I-Q_\alpha)^{-1}
\]

### Mean sojourn

\[
m(i)=\sum_j\int_0^\infty t\,Q(i,j,dt)
\]

### Long-run state occupation

\[
\pi_i^{\text{time}}
=
\frac{\pi_i m(i)}{\sum_k \pi_k m(k)}
\]

### Inverse recurrence time

\[
\frac{1}{m_{\text{return}}(i)}
=
\frac{v(i)}{\sum_k v(k)m(k)}
\]

## 24. Exercises for further work

1. Change the holding time distribution from exponential to gamma. Verify that the embedded chain is unchanged but the occupation fractions change.

2. Build a 3-state Markov renewal process where state 2 is absorbing. Estimate the probability of ever hitting state 2.

3. Numerically approximate \(R(i,j,t)\) by simulating many paths and counting visits to \(j\) before time \(t\).

4. For a finite semi-Markov process, compute the embedded stationary distribution \(\pi\), the mean sojourns \(m(i)\), and the long-run occupation distribution.

5. Compare two processes with the same embedded chain but different holding-time distributions. Which has more time in state 0? Why?